# 06 — TEMA robustness and parameter sensitivity

Same frozen 9/90/199 book. **Fit never sees OOS.** Inner-IS (last 20% of IS) is the DF neighborhood. 2022 is a *stress fold*, not a training window.

## H9 / H10

* Walk-forward years do not reverse the frozen book; nearby periods do not stably beat 9/90/199 on inner-IS.
* SL/TP and ADX/ATR grids are a plateau around 1.5 / 2.5 and ADX 20 — not a one-cell peak that wants a live retune.

Optuna stays `do_not_promote=True`. A hotter IS Sharpe that dies on DF or OOS is overfit theatre.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif (ROOT / "research").exists():
    pass
elif (ROOT / "python" / "research").exists():
    ROOT = ROOT / "python"
sys.path.insert(0, str(ROOT))
print("python root", ROOT)


In [ ]:
import pandas as pd
from research.trend_lab.data import load_symbol
from research.trend_lab.evaluate import eval_tema
from research.trend_lab.optimize import optimize_tema
from research.trend_lab.plots import df_scatter, param_heatmap
from research.trend_lab.protocol import split_frame
from research.trend_lab.tema_robust import (
    FROZEN, frozen_neighborhood, sensitivity_gates, sensitivity_periods,
    sensitivity_sl_tp, walk_forward,
)
from research.trend_lab.tema_system import TemaParams

btc, src = load_symbol("BTCUSDT", "4h")
print("BTC 4h", src, len(btc))
parts = split_frame(btc)
is_4h = parts["is"]
print("IS bars", len(is_4h), "OOS bars", len(parts["oos"]))


## Walk-forward

Each fold: frozen params, OOS window seeded with 220 IS bars from *before* that fold. No vol dial, no Optuna inside the fold.


In [ ]:
wf = walk_forward(btc, FROZEN)
display(wf.round(3))
print("2022 stress row:")
display(wf.loc[wf["stress"]].round(3) if "stress" in wf else "no stress flag")


## Sensitivity (IS only)

Frozen 9/90/199 / 1.5 / 2.5 / ADX 20 is always a row. Rank is not a license to promote the top cell.


In [ ]:
per = sensitivity_periods(is_4h)
sltp = sensitivity_sl_tp(is_4h)
gates = sensitivity_gates(is_4h)
display(per.round(3))
display(sltp.head(8).round(3))
display(gates.round(3))
param_heatmap(sltp, "sl_atr", "tp_atr", "sharpe", "IS Sharpe — SL vs TP (periods frozen)").show()
param_heatmap(sltp, "sl_atr", "tp_atr", "max_dd", "IS max DD — SL vs TP").show()
param_heatmap(gates, "min_adx", "min_atr_pct", "sharpe", "IS Sharpe — ADX vs ATR% gate").show()
print("frozen periods rank", int(per.reset_index(drop=True).index[per.reset_index(drop=True)["frozen"]].tolist()[0] + 1) if per["frozen"].any() else None)


## DF neighborhood (inner IS)

Among neighbors with train Sharpe ≥ 0.3, we want a *pool* whose val Sharpe std is small — a plateau. An empty pool or a single spike is a no-promote.


In [ ]:
dfn_p = frozen_neighborhood(is_4h, which="periods")
dfn_e = frozen_neighborhood(is_4h, which="exits")
print("periods", dfn_p["status"], "val_std", dfn_p.get("val_sharpe_std"), "n_stable", dfn_p.get("n_stable"))
print("exits  ", dfn_e["status"], "val_std", dfn_e.get("val_sharpe_std"), "n_stable", dfn_e.get("n_stable"))
if dfn_p.get("table") is not None and len(dfn_p["table"]):
    df_scatter(dfn_p["table"], "TEMA DF — fast/mid/slow (inner IS)").show()
if dfn_e.get("table") is not None and len(dfn_e["table"]):
    df_scatter(dfn_e["table"], "TEMA DF — SL/TP/ADX (inner IS)").show()


## Optuna (research only)

Run if you want the H3 replay. The winner is **not** written into the scanner. Skip this cell in a quick pass.


In [ ]:
# ot = optimize_tema(is_4h, n_trials=12, leverage=10.0)
# ot_ev = eval_tema(btc, ot["params"])
# print("do_not_promote", ot["do_not_promote"], ot.get("engine"), ot["params"])
# print("frozen IS", ot["frozen_9_90_199_is"])
# print("optuna IS", ot["is_kpis"])
# print("optuna OOS daily", ot_ev["oos_daily"])
print("Optuna cell left commented — frozen 9/90/199 is the live stack. Uncomment to replay H3.")


## Verdict rule

Promote 9/90/199 *away* only if IS Sharpe **and** DF neighborhood **and** OOS all clear for the challenger. A prettier IS heatmap is not that.
